In [2]:
# --- Importation des bibliothèques ---
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import pearsonr

In [11]:
# Charger le fichier CSV
df = pd.read_csv("../data/arbolado-publico-lineal-2017-2018.csv")

C:\Users\Augus\AppData\Local\Temp\ipykernel_72392\148030761.py:2: DtypeWarning: Columns (2,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/arbolado-publico-lineal-2017-2018.csv")


In [ ]:
# --- Charger les données ---
df = pd.read_csv("../data/arbolado-publico-lineal-2017-2018.csv")

# --- Nettoyage de la colonne calle_altura ---
def limpiar_calle_altura(valor):
    if pd.isna(valor):
        return None
    try:
        # si c’est déjà un nombre, on le garde
        return float(valor)
    except ValueError:
        # cas type: "2000-1900" ou "1900-2000"
        partes = str(valor).split('-')
        numeros = [float(p) for p in partes if p.replace('.', '', 1).isdigit()]
        if len(numeros) == 2:
            return sum(numeros) / 2   # moyenne des deux bornes
        elif len(numeros) == 1:
            return numeros[0]
        else:
            return None

df['calle_altura_num'] = df['calle_altura'].apply(limpiar_calle_altura)

df.head()

C:\Users\Augus\AppData\Local\Temp\ipykernel_72392\1590005795.py:2: DtypeWarning: Columns (2,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/arbolado-publico-lineal-2017-2018.csv")


,long,lat,nro_registro,tipo_activ,comuna,manzana,calle_nombre,calle_altura,calle_chapa,direccion_normalizada,ubicacion,nombre_cientifico,ancho_acera,estado_plantera,ubicacion_plantera,nivel_plantera,diametro_altura_pecho,altura_arbol,calle_altura_num
0,-58.378563,-34.594902,26779,Lineal,1,NaN,Esmeralda,1000.0,1120.0,ESMERALDA 1120,NaN,Tipuana tipu,5.5,Ocupada,Regular,A nivel,88.0,34.0,1000.0
1,-58.381532,-34.592319,30887,Lineal,1,NaN,Pellegrini Carlos,1300.0,1345.0,"PELLEGRINI, CARLOS 1345",Exacta,Peltophorum dubium,4.5,Ocupada,Regular,Elevada,6.0,5.0,1300.0
2,-58.379103,-34.591939,30904,Lineal,1,NaN,Arroyo,800.0,848.0,ARROYO 848,Exacta,Fraxinus pennsylvanica,4,Ocupada,Regular,A nivel,7.0,6.0,800.0
3,-58.380710,-34.591548,31336,Lineal,1,NaN,Arroyo,900.0,932.0,ARROYO 932,LD,Fraxinus pennsylvanica,NaN,Ocupada,Regular,A nivel,9.0,29.0,900.0
4,-58.380710,-34.591548,31337,Lineal,1,NaN,Arroyo,900.0,932.0,ARROYO 932,LA,Jacaranda mimosifolia,NaN,Ocupada,Regular,A nivel,13.0,8.0,900.0


In [7]:


# --- Garder uniquement les colonnes utiles ---
cols = ['calle_altura_num', 'altura_arbol', 'diametro_altura_pecho']
df = df[cols].dropna()

# --- Vérif rapide ---
print(df.head())

# --- Corrélations ---
print("\n--- Corrélations de Pearson ---")
for var in ['altura_arbol', 'diametro_altura_pecho']:
    r, p = pearsonr(df['calle_altura_num'], df[var])
    print(f"Correlation calle_altura_num vs {var}: r = {r:.3f}, p = {p:.4f}")

# --- Régressions linéaires ---
for var in ['altura_arbol', 'diametro_altura_pecho']:
    X = sm.add_constant(df['calle_altura_num'])
    y = df[var]
    model = sm.OLS(y, X).fit()
    print(f"\n--- Régression linéaire: {var} ~ calle_altura_num ---")
    print(model.summary())


   calle_altura_num  altura_arbol  diametro_altura_pecho
0            1000.0          34.0                   88.0
1            1300.0           5.0                    6.0
2             800.0           6.0                    7.0
3             900.0          29.0                    9.0
4             900.0           8.0                   13.0

--- Corrélations de Pearson ---
Correlation calle_altura_num vs altura_arbol: r = -0.000, p = 0.8253
Correlation calle_altura_num vs diametro_altura_pecho: r = -0.001, p = 0.5423

--- Régression linéaire: altura_arbol ~ calle_altura_num ---
                            OLS Regression Results                            
Dep. Variable:           altura_arbol   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                   0.04873
Date:                Thu, 23 Oct 2025   Prob (F-statistic):              0.825
Time:                